# 01 — Original policy (the reference loop)

This is the **no-intervention baseline** and the shared skeleton the other
three notebooks plug into. Read this one first: notebooks 02–04 each
describe themselves as "hook A/B/C of the loop defined in 01".

## What "original policy" means

It is a *condition*, not a reference architecture. Each backbone runs
**exactly as its own paper and released checkpoint define it** — OpenVLA as
OpenVLA, SpatialVLA as SpatialVLA, UniVLA as UniVLA. Nothing about the model
is standardised, and no weights are touched anywhere in these notebooks.

What *is* standardised is the loop around the policy:

| this notebook fixes | the policy still owns |
|---|---|
| when the observation is read | image preprocessing (resize, normalise) |
| the order actions reach `env.step` | vision encoder |
| how success is decided | LLM architecture and depth |
| how latency is counted | action tokeniser / de-tokeniser |

The policy is a black box: it only has to expose
`step(image, instruction) -> (T, action_dim)` and `reset()`. That is the
point — if two backbones are scored by different loops, a difference between
them says nothing about the backbones.

## The control loop, and the three places a method can attach

```
                    ┌─────────────────────────────────────────┐
                    │                                         │
              ┌─────▼──────┐                                  │
              │  observe   │   obs = env.step(action)         │
              └─────┬──────┘                                  │
                    │  raw camera frame (H, W, 3) uint8       │
        ╔═══════════▼═══════════╗                             │
        ║  HOOK A                ║  ← 02 foveation            │
        ║  transform the image   ║                             │
        ╚═══════════╤═══════════╝                             │
                    │                                         │
              ┌─────▼──────────────────────────────┐          │
              │  policy.step(image, instruction)   │          │
              │                                    │          │
              │   ├─ preprocess (resize/normalise) │          │
              │   ├─ vision encoder                │          │
              │   ├─ LLM decoder stack ────────────┼──╗       │
              │   └─ action de-tokenise            │  ║       │
              └─────┬──────────────────────────────┘  ║       │
                    │  actions (T, action_dim)        ║       │
        ╔═══════════▼═══════════╗            ╔════════▼═════╗ │
        ║  HOOK B                ║           ║  HOOK C      ║ │
        ║  transform the actions ║           ║  bypass      ║ │
        ║  ← 03 action repeat    ║           ║  layers      ║ │
        ╚═══════════╤═══════════╝            ║  ← 04 depth  ║ │
                    │                        ╚══════════════╝ │
              ┌─────▼──────┐                                  │
              │  env.step  │──────────────────────────────────┘
              └────────────┘
```

| hook | what it touches | when it runs | notebook |
|---|---|---|---|
| **A** | the raw camera frame, **before** the policy's own preprocessing | every control step | `02_fixed_foveation` |
| **B** | the action array the policy returned, **before** `env.step` | every control step | `03_action_repeat` |
| **C** | the decoder-layer modules inside the LLM | once per episode (calibrate), then in effect for every forward | `04_fixed_depth_pruning` |

### Why the hook points do not change with the backbone

Every one of these methods is defined at a point in the loop that **exists in
every VLA**, not at a point specific to one architecture:

* **Hook A** is defined on the *environment's* frame. Whatever the policy does
  next — SigLIP patches, a VQ tokeniser, whatever — it starts from that frame.
* **Hook B** is defined on the *action array*. Every policy returns one.
* **Hook C** is defined on a `torch.nn.ModuleList` of decoder layers. Every
  LLM-based VLA has one, though it sits at a different attribute path per
  wrapper (`04` walks candidate paths rather than hard-coding one).

That is what makes the comparison meaningful: if backbone A and backbone B are
hooked at different places, a difference in their results says nothing about
the backbones. So when porting to a new benchmark, **keep the hook points and
change only the env/policy adapters.**

## The reference loop

Deliberately plain. The hooks are present as `image_fn` / `action_fn`
parameters that default to identity, so 02 and 03 are one-line changes
rather than forks of this function.

### ⚠️ Two constants below are LIBERO conventions, not universal

`NUM_STEPS_WAIT = 10` and `DUMMY_ACTION` are taken from **OpenVLA's own
LIBERO evaluation script** (`experiments/robot/libero/run_libero_eval.py`).
They exist because LIBERO drops objects onto the table at reset, so the
first several steps issue a no-op to let the scene settle — otherwise the
policy acts on a scene mid-fall and its first actions are garbage.

**Other benchmarks do not share this.** SimplerEnv has no settle period;
CALVIN has its own reset convention and its own action layout, so
`DUMMY_ACTION`'s 7 dimensions and gripper sign may not apply at all. When
porting, take these from the target benchmark's *own* reference evaluation
rather than copying them from here. Getting the gripper sign wrong in
particular yields a policy that reaches correctly but never grasps — which
looks exactly like a method failure.

One detail that *is* universal:

* **latching `success`** — a benchmark's `done` usually means "the goal
  predicate holds *now*". If the arm nudges the object afterwards it can
  flip back, and a solved episode gets scored as a failure. Latch it and
  stop the episode there.

In [ ]:
import time
import numpy as np


# LIBERO convention, from OpenVLA's own libero eval script. NOT universal --
# see the warning above before reusing these on another benchmark.
NUM_STEPS_WAIT = 10                      # let dropped objects settle
DUMMY_ACTION = [0, 0, 0, 0, 0, 0, -1]    # no-op; -1 = gripper open in LIBERO


def identity_image(image, state):
    return image


def identity_action(actions, state):
    return actions


def run_episode(
    env,
    policy,
    instruction,
    max_steps=220,
    image_fn=identity_image,      # HOOK A
    action_fn=identity_action,    # HOOK B
    get_image=lambda obs: obs["agentview_image"],
    state=None,
):
    """One episode. Returns a dict of per-episode statistics.

    `state` is a free-form dict handed to both hooks, so a hook can keep
    per-episode state (a gaze tracker, a step counter) without this
    function knowing what the hook is.
    """
    state = {} if state is None else state
    policy.reset()

    obs = env.reset()
    success, step = False, 0
    model_time, model_calls = 0.0, 0

    while step < max_steps + NUM_STEPS_WAIT and not success:
        if step < NUM_STEPS_WAIT:
            obs, _, _, _ = env.step(DUMMY_ACTION)
            step += 1
            continue

        image = get_image(obs)

        # ---- HOOK A: transform the observation ------------------------
        policy_image = image_fn(image, state)

        t0 = time.time()
        actions = policy.step(policy_image, instruction)
        model_time += time.time() - t0
        model_calls += 1

        # ---- HOOK B: transform the actions ----------------------------
        actions = action_fn(np.asarray(actions), state)

        for row in actions:
            obs, _, done, _ = env.step(list(row))
            step += 1
            if done:
                success = True     # latch: see the note above
                break
            if step >= max_steps + NUM_STEPS_WAIT:
                break

    return {
        "success": bool(success),
        "steps": step,
        "model_calls": model_calls,
        "ms_per_call": (model_time / model_calls * 1000) if model_calls else 0.0,
        "ms_per_env_step": (model_time / max(step - NUM_STEPS_WAIT, 1)) * 1000,
    }

## Running a condition

Every condition must replay the **same initial states**. That is not a
detail: it turns each comparison into 50 matched pairs instead of two
independent samples, and a paired test (McNemar) on the same data is far
more sensitive because episodes where both conditions agree carry no
information about which is better.

In [ ]:
def run_condition(env_factory, policy, tasks, n_trials=24, **loop_kwargs):
    """Run one condition over a task list and summarise.

    env_factory(task, trial) must be deterministic in (task, trial) so that
    every condition sees the identical initial states.
    """
    episodes = []
    for task in tasks:
        for trial in range(n_trials):
            env, instruction = env_factory(task, trial)
            rec = run_episode(env, policy, instruction, **loop_kwargs)
            rec.update({"task": task, "trial": trial})
            episodes.append(rec)
            print(f"  {task} trial {trial}: "
                  f"{'SUCCESS' if rec['success'] else 'FAIL':<7} "
                  f"{rec['ms_per_call']:.0f} ms/call", flush=True)

    n_ok = sum(e["success"] for e in episodes)
    summary = {
        "n_episodes": len(episodes),
        "success_rate": n_ok / len(episodes) if episodes else 0.0,
        "avg_ms_per_call": float(np.mean([e["ms_per_call"] for e in episodes])),
        "avg_ms_per_env_step": float(np.mean([e["ms_per_env_step"] for e in episodes])),
        "avg_calls": float(np.mean([e["model_calls"] for e in episodes])),
        "episodes": episodes,      # keep per-episode records for paired tests
    }
    print(f"\n[SUMMARY] {n_ok}/{len(episodes)} = "
          f"{summary['success_rate'] * 100:.1f}%  "
          f"{summary['avg_ms_per_call']:.0f} ms/call")
    return summary

## Measuring latency without fooling yourself

Two numbers are easy to confuse:

* **ms per model call** — how long one forward costs.
* **ms per environment step** — what the robot actually experiences.

They are only the same when the policy emits one action per call. A policy that
emits a chunk of 10 and executes all of them costs `ms_per_call / 10` per
environment step. Reporting one as the other makes an already-fast policy look
slow, or vice versa.

Of the three methods here:

| method | ms per call | calls per episode |
|---|---|---|
| foveation | **unchanged** | unchanged |
| action repeat | **unchanged** | **halved** (at repeat=2) |
| depth pruning | **reduced** | unchanged |

So foveation cannot reduce latency at all (the image size, and therefore the
visual token count, is unchanged), action repeat reduces it by making fewer
calls, and depth pruning is the only one that makes a call itself cheaper.

## End-to-end check with a stub env and policy

Runs in a few milliseconds with no simulator and no checkpoint. Its purpose
is to prove the loop and the hook signatures work before any of it is
pointed at a real benchmark — so that when something does break later, the
loop is not a suspect.

In [ ]:
class StubEnv:
    """Reaches 'done' after a fixed number of steps. No physics."""

    def __init__(self, solve_at=40, size=64):
        self.solve_at, self.size, self.t = solve_at, size, 0

    def reset(self):
        self.t = 0
        return self._obs()

    def step(self, action):
        self.t += 1
        return self._obs(), 0.0, self.t >= self.solve_at, {}

    def _obs(self):
        rng = np.random.default_rng(self.t)
        return {"agentview_image":
                rng.integers(0, 256, (self.size, self.size, 3), dtype=np.uint8)}


class StubPolicy:
    """Emits a chunk of `chunk` actions per call."""

    def __init__(self, chunk=4, action_dim=7):
        self.chunk, self.action_dim = chunk, action_dim

    def reset(self):
        pass

    def step(self, image, instruction):
        return np.zeros((self.chunk, self.action_dim), dtype=np.float32)


stats = run_episode(StubEnv(), StubPolicy(), "pick up the black bowl")
print(stats)
assert stats["success"] and stats["model_calls"] > 0
print("\nloop OK")

## Porting this to another benchmark (e.g. CALVIN)

Three things are benchmark-specific; everything else in this notebook is not.

1. **`make_env(task)`** — returns something with `.reset()` and
   `.step(action)`, and a way to read the camera frame out of the observation.
2. **`make_policy(ckpt)`** — returns something with
   `.step(image, instruction) -> np.ndarray of shape (T, action_dim)`, and
   `.reset()`.
3. **The success signal** — LIBERO returns `done`; SimplerEnv returns an info
   dict; CALVIN scores completed subtasks in a sequence. Only the bookkeeping
   changes, not where the hooks go.

One caution when adapting the action convention: gripper sign and
normalisation differ per benchmark **and per checkpoint**, and getting it wrong
produces a policy that reaches correctly but never grasps — which looks like a
method failure rather than a plumbing bug.

### Validate the baseline first

Every result in the grid is a difference against "original policy", so if that
reference is wrong, every other number inherits the error. Before running any
intervention, check the no-intervention condition against the number the
backbone's **own paper** reports for that benchmark.

This is not a formality. Our OpenVLA baseline on `libero_spatial` came out at
74.0% against a published 84.7% — a systematic, reproducible 10.7-point gap
whose cause we still have not identified. Differences measured with the gap
held fixed are still usable, but absolute numbers are not comparable to the
literature until it is understood. Better to find that before the grid than
after.